#Mini-project: Gemini PDF Chatbot

In [1]:
# 1. Set Up the Environment

# Step 1: Installing all required libraries

# Installing Streamlit (for building the web interface)
!pip install streamlit

# Installing Google Generative AI (Gemini API)
!pip install google-generativeai

# Installing dotenv (to handle API keys safely)
!pip install python-dotenv

# Installing LangChain (AI framework to build the chain)
!pip install langchain

# Installing PyPDF2 (to read PDF files)
!pip install PyPDF2

# Installing chromadb (optional vector DB)
!pip install chromadb

# Installing faiss-cpu (for creating the vector store)
!pip install faiss-cpu

# Installing LangChain Google Gemini tools
!pip install langchain_google_genai

# Installing LangChain community tools
!pip install langchain-community

In [ ]:
# Step 2: Creating the .env file (in a real project, not directly in Colab)
# In Colab, we'll just set the key directly for simplicity

import os

os.environ['OPENAI_API_KEY'] = 'xxxx'

In [14]:
# 2. Extract Text from PDF Files

# Importing necessary libraries
import PyPDF2                       # To read PDF files
from io import BytesIO             # To handle uploaded files in memory
from google.colab import files     # To upload files into Colab

# Asking the user to upload one or more PDF files
uploaded_files = files.upload()

# This will store the full extracted text from all PDFs
all_text = ""

# Looping through each uploaded file
for filename in uploaded_files:
    # Reading the PDF file as binary stream
    pdf_file = BytesIO(uploaded_files[filename])

    # Creating a PDF reader object
    reader = PyPDF2.PdfReader(pdf_file)

    # Going through each page in the PDF
    for page in reader.pages:
        # Extracting text and adding it to the complete text string
        text = page.extract_text()
        if text:  # Only add if there is text on the page
            all_text += text + "\n"

# Displaying a small preview (first 1000 characters)
print(all_text[:1000])

Saving Mood Analyzer and Screening Tests for the National Center of Mental Health.pdf to Mood Analyzer and Screening Tests for the National Center of Mental Health (2).pdf
Saving An_Overview_of_the_Application_of_Sentiment_Analys.pdf to An_Overview_of_the_Application_of_Sentiment_Analys (2).pdf
International Journal of Co mputing Sciences Research (ISSN print: 2546 -0552; ISSN online: 2546 -115X)  
Vol. 6, pp. 1019 -1031  
doi: 10.25147/ijcsr.2017.001.1.88  
https://stepacademic.net  
 
 This is an Open Access article distributed under the terms of the Creative Commons Attribution License 
(http://creativecommons.org/licenses/by/4.0), which permits unrestricted use, distribution, a nd reproduction in any medium, provided the 
original work  is properly credited.  Short Paper * 
Muni -Muni: Mood Analyzer and Screening Tests for the 
National Center of Mental Health using Sentiment Analysis  
 
John Kcero C. Aujero  
College of Computer Studies , FEU – Institute of Technology , Manila, P

In [15]:
# 3. Split the Text into Chunks

# Splitting the large text into smaller overlapping chunks using RecursiveCharacterTextSplitter from langchain

from langchain.text_splitter import RecursiveCharacterTextSplitter

# Creating the splitter
# chunk_size = how big each piece should be
# chunk_overlap = how much they should overlap (helps keep context)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Splitting the text
chunks = text_splitter.split_text(all_text)

# Showing how many chunks we got and a preview
print(f"Number of chunks: {len(chunks)}")
print("\nFirst chunk preview:\n")
print(chunks[0][:500])

Number of chunks: 63

First chunk preview:

International Journal of Co mputing Sciences Research (ISSN print: 2546 -0552; ISSN online: 2546 -115X)  
Vol. 6, pp. 1019 -1031  
doi: 10.25147/ijcsr.2017.001.1.88  
https://stepacademic.net  
 
 This is an Open Access article distributed under the terms of the Creative Commons Attribution License 
(http://creativecommons.org/licenses/by/4.0), which permits unrestricted use, distribution, a nd reproduction in any medium, provided the 
original work  is properly credited.  Short Paper * 
Muni -M


In [18]:
# # 4. Generate Embeddings and Create a Vector Store

# Creating embeddings and storing them in a FAISS vector store

from langchain.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Step 1: Setting up the embedding model using the key from Step 2
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",     # Gemini's embedding model
    google_api_key=os.environ['GOOGLE_API_KEY']
)

# Step 2: Creating the FAISS vector store from the chunks
vector_store = FAISS.from_texts(chunks, embedding_model)

# Step 3: Saving the vector store locally (not needed in Colab, but for real app use)
vector_store.save_local("faiss_index")

# Optional: Checking how many vectors we stored
print("Number of vectors stored:", len(vector_store.index_to_docstore_id))

# What this does:
# Uses GoogleGenerativeAIEmbeddings to turn each chunk into a numeric vector.
# Stores all those vectors in a FAISS database.
# Saves the FAISS database locally with save_local() so you can reload it later.

Number of vectors stored: 63


In [22]:
# !pip install -U google-generativeai

In [29]:
# 5. Build the Conversational Retrieval QA Chain

# Define a Prompt + QA Chain (LangChain + OpenAI)

from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Setting up the OpenAI LLM
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",                  # or "gpt-4" if you have access
    temperature=0,
    openai_api_key=os.environ["OPENAI_API_KEY"]  # using the key you set
)

# Defining the prompt template
prompt_template = """
You are a helpful assistant. Use ONLY the context below to answer the question.
If you don’t know the answer, say “I don't know.”

Context:
{context}

Question:
{question}
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

# Creating the QA chain with FAISS retriever
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

<ipython-input-29-bbf33584698b>:10: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


In [30]:
# import google.generativeai as genai
# import os

# # setting your API key (already done in Step 2)
# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# # loading a supported model for your account
# model = genai.GenerativeModel("models/text-bison-001")

# # test that the model works
# response = model.generate_content("Hello! What can you do?")
# print(response.text)

In [32]:
# Asking a test question
question = "What is the document about?"
answer = qa_chain.run(question)

print("Q:", question)
print("A:", answer)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}